# Ebaluazioa 3: Klasifikazio Probabilistikoa (MNIST) - Erantzun Ezezagunei Aurre Eginez
Helburua modeloei erantzuteko eskatzea denez aurrez _inoiz_ ikusi ez duten klase baten aurrean (adb. 3 zenbakia ezabatuko dugu trebakuntzatik), entropia erabiliko da ziurgabetasuna gisa.

In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.generators import MNISTGenerator
from src.noise import AbsoluteGaussianNoise
from src.shadows import ClassShadow
from src.classifiers import NeuralNetworkMCDropoutClassifier, RandomForestClassifierModel, BayesianNeuralNetworkClassifier
from src.splitter import GapSplitter
from src.orchestrator import ClassificationOrchestrator
from src.visualization import plot_mnist_classification_results, plot_mnist_sample_uncertainties

In [ ]:
N_SAMPLES = 5000 
HIDDEN_CLASS = 3
EPOCHS = 20

generator = MNISTGenerator(train=True)
shadow = ClassShadow(shadow_class=HIDDEN_CLASS)
splitter = GapSplitter(test_size=0.2, random_state=42)
noise_strategy = AbsoluteGaussianNoise(std_dev=0.0)

In [ ]:
models = [
    RandomForestClassifierModel(n_estimators=50, max_depth=15, random_state=42),
    NeuralNetworkMCDropoutClassifier(hidden_dim=128, num_classes=10, dropout_rate=0.3, epochs=EPOCHS, n_iter=30),
    BayesianNeuralNetworkClassifier(hidden_dim=128, num_classes=10, epochs=EPOCHS, kl_weight=0.01)
]

In [ ]:
print(f"\n{'='*60}")
print(f"EBALUAZIOA: MNIST datu-basea ({HIDDEN_CLASS} Klasea Ezkutatuz)")
print(f"{'='*60}")

for m in models:
    print(f"\n-> Modeloa entropiarekin: {m.get_name()}")

    orchestrator = ClassificationOrchestrator(generator, splitter, m)
    artifacts = orchestrator.run(
        n_samples=N_SAMPLES,
        x_range=(0,0), 
        noise_strategy=noise_strategy,
        shadow=shadow
    )

    if artifacts.uncertainties is not None:
        ent_vis = np.mean(artifacts.uncertainties['visible'])
        ent_sha = np.mean(artifacts.uncertainties['shadow'])
        print(f"    * Zehaztasuna (Ikusgai): {artifacts.metrics.acc_test_visible:.4f}")
        print(f"    * Zehaztasuna (Itzala) : {artifacts.metrics.acc_test_shadow:.4f}")
        print(f"    * Entropia (Ikusgai)   : {ent_vis:.4f}")
        print(f"    * Entropia (Itzala)    : {ent_sha:.4f}")

    plot_mnist_classification_results(artifacts.split_data, artifacts.predictions, artifacts.uncertainties, title=f"Eredua: {m.get_name()}")
    plot_mnist_sample_uncertainties(artifacts.split_data, artifacts.predictions, artifacts.uncertainties, num_samples=5)
    
print("\nEsperimentua Arrakastaz Bideratua")